# Weather Download (NOAA ISD-Lite, 2024)

Download station mapping and raw hourly weather data for top-50 airports.

## Outputs
- `data/raw/weather/noaa_isd_history.csv`
- `data/raw/weather/weather_station_map_top50_2024.csv`
- `data/raw/weather/weather_2024_isd_lite_raw.parquet`
- `data/reports/weather/weather_download_summary_2024.csv`


In [4]:
import gzip
import io
import os
from pathlib import Path

import certifi
import pandas as pd
import requests
from tqdm import tqdm

YEAR = 2024

# Project-root aware path setup (works in local + Colab)
def resolve_project_root() -> Path:
    env_root = os.getenv("FLIGHT_PROJECT_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    return cwd

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
RAW_DIR = DATA_ROOT / "raw" / "weather"
REPORT_DIR = DATA_ROOT / "reports" / "weather"
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TOP_50_AIRPORTS = [
    'ATL','LAX','ORD','DFW','DEN','JFK','SFO','SEA','LAS','MCO',
    'EWR','CLT','PHX','IAH','MIA','BOS','MSP','FLL','DTW','PHL',
    'LGA','BWI','SLC','DCA','SAN','MDW','TPA','HNL','PDX','STL',
    'DAL','BNA','AUS','IAD','OAK','MSY','RDU','SMF','SJC','SNA',
    'MCI','SAT','RSW','CLE','PIT','IND','CMH','OGG','ABQ','BUR'
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Raw dir:      {RAW_DIR}")
print(f"Reports dir:  {REPORT_DIR}")
print(f"Top airports: {len(TOP_50_AIRPORTS)}")


Project root: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450
Data root:    /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data
Raw dir:      /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/raw/weather
Reports dir:  /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/reports/weather
Top airports: 50


## Step 1: Build station mapping (IATA -> USAF/WBAN)

In [6]:
isd_url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"

# Use requests + certifi for more robust SSL handling on local Python.
try:
    resp = requests.get(isd_url, timeout=60, verify=certifi.where())
    resp.raise_for_status()
except requests.exceptions.SSLError as e:
    raise RuntimeError(
        "SSL verification failed while downloading isd-history.csv. "
        "If this persists, run Python's certificate installer or use your project venv."
    ) from e

isd_history = pd.read_csv(io.StringIO(resp.text), low_memory=False)
isd_history.to_csv(RAW_DIR / "noaa_isd_history.csv", index=False)

# Keep US stations with valid 4-letter ICAO codes.
# Do NOT restrict to prefix 'K' only: Hawaii/Alaska use other prefixes (e.g., PHNL, PHOG, PANC).
us_stations = isd_history[
    (isd_history["CTRY"] == "US") &
    (isd_history["ICAO"].notna()) &
    (isd_history["ICAO"].astype(str).str.len() == 4)
].copy()

# Default mapping: ICAO starts with K -> IATA is ICAO without first letter.
# This covers most continental US airports.
us_stations["IATA"] = us_stations["ICAO"].str[1:]

# Explicit ICAO -> IATA overrides for airports where the default rule does not apply.
icao_to_iata_overrides = {
    "PHNL": "HNL",  # Honolulu
    "PHOG": "OGG",  # Kahului (Maui)
}
override_mask = us_stations["ICAO"].isin(icao_to_iata_overrides)
us_stations.loc[override_mask, "IATA"] = us_stations.loc[override_mask, "ICAO"].map(icao_to_iata_overrides)

us_stations["END"] = pd.to_datetime(us_stations["END"], errors="coerce")

station_map = (
    us_stations[us_stations["IATA"].isin(TOP_50_AIRPORTS)]
    .sort_values("END", ascending=False)
    .groupby("IATA", as_index=False)
    .first()[["IATA", "USAF", "WBAN", "ICAO"]]
)

station_map["USAF"] = station_map["USAF"].astype(str).str.zfill(6)
station_map["WBAN"] = station_map["WBAN"].astype(str).str.zfill(5)
station_map["station_id"] = station_map["USAF"] + station_map["WBAN"]
station_map.to_csv(RAW_DIR / "weather_station_map_top50_2024.csv", index=False)

missing = sorted(set(TOP_50_AIRPORTS) - set(station_map["IATA"]))
print(f"Mapped airports: {len(station_map)}")
print(f"Missing airports: {missing}")
print(f"Applied ICAO overrides: {icao_to_iata_overrides}")
station_map.head()


Mapped airports: 50
Missing airports: []
Applied ICAO overrides: {'PHNL': 'HNL', 'PHOG': 'OGG'}


,IATA,USAF,WBAN,ICAO,station_id
0,ABQ,723650,23050,KABQ,72365023050
1,ATL,722190,13874,KATL,72219013874
2,AUS,722540,13904,KAUS,72254013904
3,BNA,723270,13897,KBNA,72327013897
4,BOS,725090,14739,KBOS,72509014739


## Step 2: Download raw ISD-Lite data for mapped airports

In [7]:
ISD_COLS = [
    "year", "month", "day", "hour",
    "air_temp", "dew_point", "sea_level_pres", "wind_dir",
    "wind_speed", "sky_cover", "precip_1h", "precip_6h"
]

def download_isd_lite(usaf: str, wban: str, year: int = 2024) -> pd.DataFrame | None:
    url = f"https://www.ncei.noaa.gov/pub/data/noaa/isd-lite/{year}/{usaf}-{wban}-{year}.gz"
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with gzip.open(io.BytesIO(resp.content), "rt") as f:
            return pd.read_fwf(f, header=None, names=ISD_COLS)
    except Exception:
        return None

weather_dfs = []
failed = []
for _, row in tqdm(station_map.iterrows(), total=len(station_map), desc="Downloading weather"):
    wdf = download_isd_lite(row["USAF"], row["WBAN"], year=YEAR)
    if wdf is None:
        failed.append((row["IATA"], row["USAF"], row["WBAN"]))
        continue
    wdf["IATA"] = row["IATA"]
    weather_dfs.append(wdf)

if not weather_dfs:
    raise RuntimeError("No weather data downloaded.")

weather_raw = pd.concat(weather_dfs, ignore_index=True)
out_path = RAW_DIR / "weather_2024_isd_lite_raw.parquet"
weather_raw.to_parquet(out_path, index=False)

summary = pd.DataFrame([
    {
        "year": YEAR,
        "target_airports": len(TOP_50_AIRPORTS),
        "mapped_airports": len(station_map),
        "missing_airports": len(set(TOP_50_AIRPORTS) - set(station_map["IATA"])),
        "successful_station_downloads": len(weather_dfs),
        "failed_station_downloads": len(failed),
        "weather_rows": int(len(weather_raw)),
    }
])
summary_path = REPORT_DIR / "weather_download_summary_2024.csv"
summary.to_csv(summary_path, index=False)

print(f"Weather rows: {len(weather_raw):,}")
print(f"Saved: {out_path}")
print(f"Failed stations: {len(failed)}")
print(f"Saved summary: {summary_path}")
if failed:
    print("First failures:", failed[:5])


Weather rows: 438,299
Saved: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/raw/weather/weather_2024_isd_lite_raw.parquet
Failed stations: 0
Saved summary: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/reports/weather/weather_download_summary_2024.csv
